# Aggregate reserve (lithology-based)

Per-cell reserve from physical parameters:

$$R \,[\mathrm{Mt}] \;=\; A \,[\mathrm{km}^2] \;\times\; T \,[\mathrm{m}] \;\times\; \rho \,[\mathrm{t/m^3}] \;\times\; r$$

- `A` = **mineable land** (km² per 1-km cell), single file from `2-2-estimate-mineable-land.ipynb` (τ = OSM P50, empirically anchored).
- `T` = extractable thickness (m), lithology-specific, scaled by L/M/H.
- `rho` = in-situ density (t/m³), lithology-specific.
- `r` = recovery rate (fraction), lithology-specific, shifted by L/M/H.

Lithology from GLiM (16 classes). Evaporite, water, and ice are excluded (R = 0).

## Terminology

| Layer | Definition | File |
|---|---|---|
| Available land | Categorical exclusions only | `available-land-1km.tif` |
| Mineable land | Available × `suitability ≥ P50` | `mineable-land-1km.tif` (single file) |
| **Aggregate reserve** | **Mineable land × `T·ρ·r`** | **`aggregate-reserve-1km-{L,M,H}.tif`** |

In CRIRSCO/JORC terms: an **accessible / mineable reserve** — geologically present *and* technically feasible.

## L / M / H scenarios (lithology only)

Mineable land is **fixed** (calibrated at OSM P50). L/M/H reflects only the engineering uncertainty in extractability parameters (T, r):

| Scenario | T (per class) | r (per class) | Lithology factor vs M |
|---|---|---|---|
| **L** (conservative) | `0.5 · T_M` | `r_M − 0.10` (floor 0.50) | ~0.43× |
| **M** (central) | `T_M` | `r_M` | 1.0× |
| **H** (optimistic) | `1.5 · T_M` | `r_M + 0.05` (cap 0.85) | ~1.60× |

Expected reserve spread: **L:M:H ≈ 1 : 2.3 : 3.7**.

Outputs: `aggregate-reserve-{1km,50km}-{L|M|H}.tif`, values in Mt per cell.

In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from rasterio.warp import reproject, Resampling
from tqdm import tqdm
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'

## Load inputs

A single `mineable-land-1km.tif` file (produced by `2-2`) is loaded once and reused across L/M/H. Antarctica exclusion and suitability filter are already baked in.

In [3]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')
country_raster_50km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-50km.tif')

mineable_land_1km = rio.open(RESULTS / 'mining-area' / 'mineable-land-1km.tif').read(1)
print(f'Mineable land: {mineable_land_1km.sum() / 1e6:.2f} M km^2')

Mineable land: 3.82 M km^2


In [4]:
# Reproject GLiM to the 1-km Eckert IV grid (nearest, to preserve class codes).
GLiM_ = rio.open(DATA / 'GLiM.tif')
GLiM_src = GLiM_.read(1)

GLiM_1km = np.zeros(country_raster_1km_.shape, dtype=np.uint8)
reproject(
    source=GLiM_src,
    destination=GLiM_1km,
    src_transform=GLiM_.transform,
    src_crs=GLiM_.crs,
    dst_transform=country_raster_1km_.transform,
    dst_crs=country_raster_1km_.crs,
    resampling=Resampling.nearest,
)
del GLiM_src

## Lithology parameter table

Central values (`T_M`, `rho`, `r_M`) per GLiM class. Scenario shifts:
- `T_L = 0.5 * T_M`, `T_H = 1.5 * T_M`
- `r_L = max(0.50, r_M - 0.10)`, `r_H = min(0.85, r_M + 0.05)`

Classes excluded (R = 0): `ev` (evaporite), `wb` (water), `ig` (ice). Class `nd` (no data) and code 0 (off-grid) use a fallback central value.

These values are engineering-grade defaults. Replace with cited values before publication (see chat history for suggested references: Telford 1990 / Carmichael 1989 for densities, Hartman & Mutmansky 2002 / SME Handbook 2011 for recoveries, Langer 1988 / Bliss et al. 2012 for the methodological framework).

In [5]:
# code: GLiM integer class. name: short tag. type: aggregate category.
# T_M (m), rho (t/m^3), r_M (recovery fraction at central scenario).
params = pd.DataFrame([
    # code, name, type,                       T_M, rho,  r_M
    ( 1, 'su', 'sand & gravel',                10, 1.80, 0.70),
    ( 2, 'vb', 'basic volcanic',               40, 2.90, 0.75),
    ( 3, 'ss', 'siliciclastic sedimentary',    30, 2.40, 0.65),
    ( 4, 'pb', 'basic plutonic',               50, 2.95, 0.80),
    ( 5, 'sm', 'mixed sedimentary',            30, 2.50, 0.65),
    ( 6, 'sc', 'carbonate sedimentary',        40, 2.60, 0.75),
    ( 7, 'va', 'acid volcanic',                40, 2.55, 0.75),
    ( 8, 'mt', 'metamorphic',                  40, 2.75, 0.75),
    ( 9, 'pa', 'acid plutonic',                50, 2.70, 0.80),
    (10, 'vi', 'intermediate volcanic',        40, 2.70, 0.75),
    (11, 'wb', 'water (excluded)',              0, 0.00, 0.00),
    (12, 'py', 'pyroclastic',                  20, 2.20, 0.60),
    (13, 'pi', 'intermediate plutonic',        50, 2.80, 0.80),
    (14, 'ev', 'evaporite (excluded)',          0, 0.00, 0.00),
    (15, 'nd', 'no data (fallback)',           30, 2.50, 0.70),
    (16, 'ig', 'ice (excluded)',                0, 0.00, 0.00),
], columns=['code', 'name', 'type', 'T_M', 'rho', 'r_M'])

params['T_L'] = 0.5 * params['T_M']
params['T_H'] = 1.5 * params['T_M']
params['r_L'] = np.where(params['r_M'] > 0, np.maximum(0.50, params['r_M'] - 0.10), 0.0)
params['r_H'] = np.where(params['r_M'] > 0, np.minimum(0.85, params['r_M'] + 0.05), 0.0)
params

,code,name,type,T_M,rho,r_M,T_L,T_H,r_L,r_H
0,1,su,sand & gravel,10,1.80,0.70,5.0,15.0,0.60,0.75
1,2,vb,basic volcanic,40,2.90,0.75,20.0,60.0,0.65,0.80
2,3,ss,siliciclastic sedimentary,30,2.40,0.65,15.0,45.0,0.55,0.70
3,4,pb,basic plutonic,50,2.95,0.80,25.0,75.0,0.70,0.85
4,5,sm,mixed sedimentary,30,2.50,0.65,15.0,45.0,0.55,0.70
5,6,sc,carbonate sedimentary,40,2.60,0.75,20.0,60.0,0.65,0.80
6,7,va,acid volcanic,40,2.55,0.75,20.0,60.0,0.65,0.80
7,8,mt,metamorphic,40,2.75,0.75,20.0,60.0,0.65,0.80
8,9,pa,acid plutonic,50,2.70,0.80,25.0,75.0,0.70,0.85
9,10,vi,intermediate volcanic,40,2.70,0.75,20.0,60.0,0.65,0.80


In [6]:
# Build per-class (T * rho * r) lookup for each scenario.
# Index 0 (off-grid / unclassified) is set to 0, so reserve is 0 there.
def make_factor_lookup(T_col: str, r_col: str) -> np.ndarray:
    lookup = np.zeros(17, dtype=np.float32)  # codes 0..16
    for _, row in params.iterrows():
        lookup[int(row['code'])] = float(row[T_col] * row['rho'] * row[r_col])
    return lookup

factor_lookups = {
    'L': make_factor_lookup('T_L', 'r_L'),  # conservative (recommended primary estimate)
    'M': make_factor_lookup('T_M', 'r_M'),  # central
    'H': make_factor_lookup('T_H', 'r_H'),  # optimistic
}

# Sanity check: Mt per km^2 for each class under each scenario.
pd.DataFrame({
    'name': params['name'].values,
    'L_Mt_per_km2': [factor_lookups['L'][int(c)] for c in params['code']],
    'M_Mt_per_km2': [factor_lookups['M'][int(c)] for c in params['code']],
    'H_Mt_per_km2': [factor_lookups['H'][int(c)] for c in params['code']],
})

,name,L_Mt_per_km2,M_Mt_per_km2,H_Mt_per_km2
0,su,5.400000,12.600000,20.250000
1,vb,37.700001,87.000000,139.199997
2,ss,19.799999,46.799999,75.599998
3,pb,51.625000,118.000000,188.062500
4,sm,20.625000,48.750000,78.750000
5,sc,33.799999,78.000000,124.800003
6,va,33.150002,76.500000,122.400002
7,mt,35.750000,82.500000,132.000000
8,pa,47.250000,108.000000,172.125000
9,vi,35.099998,81.000000,129.600006


In [7]:
def output_tif(path, array, template_raster):
    """Write a 2D array to GeoTIFF following the metadata of a template raster."""
    with rio.open(
        path,
        'w',
        driver='GTiff',
        height=array.shape[0],
        width=array.shape[1],
        count=1,
        dtype=array.dtype,
        crs=template_raster.crs,
        transform=template_raster.transform,
        compress='lzw',
    ) as dst:
        dst.write(array, 1)

## Compute reserve per scenario

For each scenario (L / M / H):
1. Build the per-cell lithology factor `f[i,j] = lookup[GLiM_1km[i,j]]` (Mt per km²) for that scenario.
2. `reserve_1km = mineable_land_1km * f` (mineable land is shared across L/M/H).
3. Sum-resample to 50 km.

In [8]:
(RESULTS / 'mining-area').mkdir(parents=True, exist_ok=True)

for scenario, lookup in tqdm(factor_lookups.items(), desc='scenario'):
    factor_1km  = lookup[GLiM_1km]                                       # Mt per km^2
    reserve_1km = (mineable_land_1km * factor_1km).astype(np.float32)    # Mt per cell

    reserve_50km = np.zeros(country_raster_50km_.shape, dtype=np.float32)
    reproject(
        source=reserve_1km, destination=reserve_50km,
        src_transform=country_raster_1km_.transform, src_crs=country_raster_1km_.crs,
        dst_transform=country_raster_50km_.transform, dst_crs=country_raster_50km_.crs,
        resampling=Resampling.sum,
    )

    path_1km  = RESULTS / 'mining-area' / f'aggregate-reserve-1km-{scenario}.tif'
    path_50km = RESULTS / 'mining-area' / f'aggregate-reserve-50km-{scenario}.tif'
    output_tif(path_1km,  reserve_1km,  country_raster_1km_)
    output_tif(path_50km, reserve_50km, country_raster_50km_)

    print(f'{scenario}: reserve {reserve_1km.sum()/1e6:.2f} Tt '
          f'(non-zero cells = {(reserve_1km > 0).sum():,})')

scenario:  33%|███▎      | 1/3 [03:26<06:53, 206.53s/it]

L: reserve 94.88 Tt (non-zero cells = 4,065,133)


scenario:  67%|██████▋   | 2/3 [06:34<03:15, 195.53s/it]

M: reserve 220.12 Tt (non-zero cells = 4,065,133)


scenario: 100%|██████████| 3/3 [09:49<00:00, 196.48s/it]

H: reserve 352.95 Tt (non-zero cells = 4,065,133)


## Quick sanity check: per-country totals (conservative scenario)

Reads the freshly written `L` raster (the recommended primary estimate) and sums by country, with country ISO codes merged in for readability.

In [9]:
pseudo_country_codes = pd.read_csv(DATA / 'country-pseudo-values.csv')

country_50km   = country_raster_50km_.read(1)
reserve_L_50km = rio.open(RESULTS / 'mining-area' / 'aggregate-reserve-50km-L.tif').read(1)

valid = country_50km > 0
countries, idx = np.unique(country_50km[valid], return_inverse=True)
totals = np.bincount(idx, weights=reserve_L_50km[valid].astype(np.float64))
summary = (
    pd.DataFrame({'country_code': countries.astype(int), 'reserve_Mt': totals})
      .merge(
          pseudo_country_codes[['value', 'country', 'full name']],
          left_on='country_code', right_on='value', how='left',
      )
      .drop(columns='value')
      .sort_values('reserve_Mt', ascending=False)
      .reset_index(drop=True)
)
summary.head(20)

,country_code,reserve_Mt,country,full name
0,145,1.206798e+07,RUS,Russia
1,94,1.130150e+07,USA,United States
2,114,7.631717e+06,FRA,France
3,149,3.542696e+06,FIN,Finland
4,101,3.373403e+06,DEU,Germany
5,59,3.301095e+06,BRA,Brazil
6,232,3.234161e+06,ITA,Italy
7,134,3.165262e+06,SWE,Sweden
8,235,2.968726e+06,ESP,Spain
9,246,2.774681e+06,CAN,Canada


In [10]:
# Sanity check: global reserve per scenario.
for s in ['L', 'M', 'H']:
    a = rio.open(RESULTS / 'mining-area' / f'aggregate-reserve-1km-{s}.tif').read(1).sum() / 1e6
    print(f'  {s}: {a:.2f} Tt')

  L: 94.88 Tt
  M: 220.12 Tt
  H: 352.95 Tt
